# Effect Sizes and Reporting

## Overview

Effect sizes quantify the magnitude of a difference independently of sample size. They are essential for communicating practical significance, enabling meta-analysis, and designing future studies.

**Standardised effect sizes:**

| Outcome | Effect size | Formula | Benchmark (small/med/large) |
|---|---|---|---|
| Continuous | Cohen's d | (μ₁−μ₂)/σ_pooled | 0.2 / 0.5 / 0.8 |
| Continuous | Hedges' g | Bias-corrected d | same |
| Binary | Odds ratio | (p₁/(1−p₁))/(p₂/(1−p₂)) | 1.5 / 2.5 / 4.0 |
| Binary | Risk ratio | p₁/p₂ | — |
| Binary | NNT | 1/ARR | — |
| Non-parametric | Rank-biserial r | 2U/(n₁n₂)−1 | 0.1 / 0.3 / 0.5 |
| Correlation | r / R² | — | 0.1 / 0.3 / 0.5 |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
n   = 60
control   = rng.normal(15.0, 5.0, n)
treatment = rng.normal(18.0, 5.2, n)
print("Effect size calculations for restoration experiment")
print(f"Control:   mean={control.mean():.2f}, sd={control.std():.2f}")
print(f"Treatment: mean={treatment.mean():.2f}, sd={treatment.std():.2f}")

---
## Standardised Effect Sizes

In [ ]:
def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    s_pooled = np.sqrt(((n1-1)*a.std()**2 + (n2-1)*b.std()**2) / (n1+n2-2))
    return (a.mean() - b.mean()) / s_pooled

def hedges_g(a, b):
    d = cohens_d(a, b)
    n1, n2 = len(a), len(b)
    # Correction factor J
    df = n1 + n2 - 2
    J  = 1 - 3 / (4*df - 1)
    return d * J

def rank_biserial(a, b):
    u, _ = stats.mannwhitneyu(a, b, alternative='two-sided')
    return 2*u / (len(a)*len(b)) - 1

d   = cohens_d(treatment, control)
g   = hedges_g(treatment, control)
r   = rank_biserial(treatment, control)
_, p = stats.ttest_ind(treatment, control)
print(f"Cohen's d:         {d:.3f}  (small=0.2, medium=0.5, large=0.8)")
print(f"Hedges' g:         {g:.3f}  (bias-corrected for small samples)")
print(f"Rank-biserial r:   {r:.3f}  (non-parametric, no normality needed)")
print(f"p-value:           {p:.4f}")

# Visualise effect size as distribution overlap
x = np.linspace(0, 35, 300)
fig, ax = plt.subplots(figsize=(9,4))
ax.fill_between(x, stats.norm.pdf(x, control.mean(),  control.std()),
                alpha=0.4, color='steelblue', label='Control')
ax.fill_between(x, stats.norm.pdf(x, treatment.mean(), treatment.std()),
                alpha=0.4, color='#e74c3c', label='Treatment')
ax.axvline(control.mean(),   color='steelblue', lw=2, linestyle='--')
ax.axvline(treatment.mean(), color='#e74c3c',   lw=2, linestyle='--')
ax.set_xlabel('Species richness'); ax.set_ylabel('Density')
ax.set_title(f"Distribution Overlap  d={d:.2f}, g={g:.2f}")
ax.legend(); plt.tight_layout(); plt.show()

---
## Binary Outcome Effect Sizes

In [ ]:
# Binary outcomes: site restoration success
p_ctrl, p_trt = 0.35, 0.55
n_b = 80

# Compute from sample data
s_ctrl = rng.binomial(1, p_ctrl, n_b)
s_trt  = rng.binomial(1, p_trt,  n_b)
p_c_obs = s_ctrl.mean()
p_t_obs = s_trt.mean()

ARR = p_t_obs - p_c_obs                           # Absolute Risk Reduction
RR  = p_t_obs / (p_c_obs + 1e-8)                  # Relative Risk
OR  = (p_t_obs/(1-p_t_obs)) / (p_c_obs/(1-p_c_obs+1e-8))  # Odds Ratio
NNT = 1 / (ARR + 1e-8)                            # Number Needed to Treat

print(f"Binary effect size summary:")
print(f"  p_control (obs):   {p_c_obs:.3f}")
print(f"  p_treatment (obs): {p_t_obs:.3f}")
print(f"  ARR:  {ARR:.3f}  ({ARR*100:.1f} percentage points)")
print(f"  RR:   {RR:.3f}  (treatment {(RR-1)*100:.0f}% more likely to succeed)")
print(f"  OR:   {OR:.3f}")
print(f"  NNT:  {NNT:.1f}  (need to treat {NNT:.0f} sites to get 1 extra success)")

# 95% CI for OR using log-normal approximation
log_OR = np.log(OR)
se_logOR = np.sqrt(1/(s_trt.sum()+0.5) + 1/(n_b-s_trt.sum()+0.5)
                   + 1/(s_ctrl.sum()+0.5) + 1/(n_b-s_ctrl.sum()+0.5))
ci_lo_or = np.exp(log_OR - 1.96*se_logOR)
ci_hi_or = np.exp(log_OR + 1.96*se_logOR)
print(f"  OR 95% CI: [{ci_lo_or:.3f}, {ci_hi_or:.3f}]")

---
## Confidence Intervals for Effect Sizes

In [ ]:
# Bootstrap CI for Cohen's d
def bootstrap_d_ci(a, b, n_boot=5000, ci=0.95):
    diffs = []
    for _ in range(n_boot):
        a_b = rng.choice(a, len(a), replace=True)
        b_b = rng.choice(b, len(b), replace=True)
        diffs.append(cohens_d(a_b, b_b))
    lo = np.percentile(diffs, (1-ci)/2*100)
    hi = np.percentile(diffs, (1+ci)/2*100)
    return lo, hi, np.array(diffs)

d_lo, d_hi, d_boot = bootstrap_d_ci(treatment, control)
print(f"Cohen's d = {d:.3f}  Bootstrap 95% CI: [{d_lo:.3f}, {d_hi:.3f}]")

# Forest plot: multiple effect size measures
measures = {
    "Cohen's d":          (d,   d_lo, d_hi,   0),
    "Hedges' g":          (g,   d_lo*g/d, d_hi*g/d, 0),
    "Rank-biserial r":    (r,   None, None, 1),   # different scale
}
fig, ax = plt.subplots(figsize=(9,3))
for i, (name, (est, lo, hi, _)) in enumerate(measures.items()):
    if lo is not None:
        ax.plot([lo, hi], [i,i], '-', color='steelblue', lw=3)
    ax.plot(est, i, 'o', color='#e74c3c', ms=10, zorder=3)
    label = f'{est:.3f}' if lo is None else f'{est:.3f} [{lo:.3f}, {hi:.3f}]'
    ax.text(1.0, i, label, va='center', fontsize=9)
ax.axvline(0, color='black', lw=1, linestyle='--')
ax.set_yticks(range(len(measures))); ax.set_yticklabels(list(measures.keys()))
ax.set_xlabel('Effect size'); ax.set_xlim(-0.3, 1.5)
ax.set_title('Effect Sizes with Bootstrap CIs')
plt.tight_layout(); plt.show()

In [ ]:
# Complete results reporting template
_, p_val = stats.ttest_ind(treatment, control)
t_stat, _ = stats.ttest_ind(treatment, control)
df_val = len(treatment) + len(control) - 2
diff   = treatment.mean() - control.mean()
se_diff = np.sqrt(treatment.var()/len(treatment) + control.var()/len(control))
ci95   = stats.t.ppf(0.975, df_val) * se_diff

print("=" * 65)
print("COMPLETE RESULTS REPORT")
print("=" * 65)
print(f"\nComparison: Restored (n={len(treatment)}) vs Control (n={len(control)}) sites")
print(f"\nDescriptive statistics:")
print(f"  Control:   M={control.mean():.2f}, SD={control.std():.2f}, n={len(control)}")
print(f"  Treatment: M={treatment.mean():.2f}, SD={treatment.std():.2f}, n={len(treatment)}")
print(f"\nInferential statistics:")
print(f"  t({df_val}) = {t_stat:.3f}, p = {p_val:.4f}")
print(f"  Mean difference: {diff:.3f} (95% CI: {diff-ci95:.3f} to {diff+ci95:.3f})")
print(f"\nEffect size:")
print(f"  Cohen's d = {d:.3f} (medium effect)")
print(f"  Bootstrap 95% CI for d: [{d_lo:.3f}, {d_hi:.3f}]")
print(f"\nInterpretation:")
sig = 'statistically significant' if p_val < 0.05 else 'not statistically significant'
print(f"  The difference was {sig} (α=0.05).")
print(f"  The effect size (d={d:.2f}) suggests a medium practical difference.")

---

## Common Pitfalls

**1. Reporting only p-values without effect sizes**  
A p-value tells you whether the data are consistent with zero effect — it does not quantify how large the effect is. With large samples, trivially small differences produce p < 0.001. Always report the estimated effect size and its confidence interval alongside the p-value.

**2. Using Cohen's d benchmarks (small/medium/large) as universal standards**  
Cohen's benchmarks were intended as rough defaults for the social sciences in the absence of domain knowledge, not as universal standards. A Cohen's d of 0.2 may be enormous in one context (pharmacology) and negligible in another (ecology). Calibrate "small" and "large" against what matters in your domain.

**3. Confusing relative risk and odds ratio — especially when baseline rates are high**  
OR and RR are numerically similar when baseline rates are low (<10%), but diverge substantially at higher rates. An OR of 3.0 when p_control=0.5 corresponds to RR=1.5 — very different practical meanings. Always report both OR and the absolute risk difference (ARR) for binary outcomes.

**4. Not bootstrapping CI for effect sizes — relying on asymptotic approximations with small n**  
Analytic CIs for Cohen's d assume normality and are inaccurate for n < 30. Bootstrap CIs make no distributional assumptions and are preferred for non-normal data or small samples. Use `scipy.stats.bootstrap` or implement the percentile bootstrap directly.

**5. Reporting effect size without its confidence interval**  
A point estimate of d=0.5 is uninterpretable without a CI. With n=10 per group, the 95% CI might span [-0.2, 1.2] — entirely consistent with zero effect. The CI communicates estimation uncertainty and is required for any meaningful interpretation of effect magnitude.
---
*python_methods_library - Samantha McGarrigle*